# Milestone 1 — Data Acquisition & Cleaning

Notebook ini menyelesaikan 4 tahap kurasi data: akuisisi/ingestion, audit format & skema, imputasi rasional, serta encoding & scaling dengan `ColumnTransformer`.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path('healthcare-dataset-stroke-data.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Akuisisi & Ingestion
Dataset dimuat sebagai `DataFrame` Pandas dari file CSV.

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.info()

## 2. Audit Format & Schema
Kolom dinormalisasi menjadi lowercase dan whitespace pada kolom teks dihapus. Audit menunjukkan tidak ada baris duplikat. `smoking_status = Unknown` dipertahankan sebagai kategori eksplisit karena bukan nilai `NaN`.

In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype('string').str.strip()
print('Duplicate rows:', df.duplicated().sum())
print('Missing values before imputation:')
print(df.isna().sum())

## 3. Imputasi Rasional
`bmi` adalah satu-satunya kolom dengan missing value: 201 baris (~3,9%). Median dipilih karena robust terhadap outlier dan tidak mengurangi jumlah observasi. Kategori `Unknown` pada `smoking_status` tidak diubah menjadi modus agar informasi ketidakpastian tetap terjaga.

In [ ]:
bmi_median = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(bmi_median)
for c in ['id','hypertension','heart_disease','stroke']:
    df[c] = df[c].astype(int)
for c in ['age','avg_glucose_level','bmi']:
    df[c] = df[c].astype(float)
print('BMI median used:', bmi_median)
print('Missing values after imputation:')
print(df.isna().sum())

## 4. Encoding & Scaling
Fitur kontinu (`age`, `avg_glucose_level`, `bmi`) distandardisasi dengan `StandardScaler`. Fitur biner (`hypertension`, `heart_disease`) dipertahankan 0/1. Fitur nominal di-One-Hot Encode. `handle_unknown="ignore"` menjaga pipeline tetap aman ketika data baru memiliki kategori yang tidak muncul saat fitting.

In [ ]:
X = df.drop(columns=['stroke','id'])
y = df['stroke']
num_cols = ['age','avg_glucose_level','bmi']
bin_cols = ['hypertension','heart_disease']
cat_cols = ['gender','ever_married','work_type','residence_type','smoking_status']
preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('bin', 'passthrough', bin_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
], remainder='drop', verbose_feature_names_out=False)
X_processed = preprocess.fit_transform(X)
processed = pd.DataFrame(X_processed, columns=preprocess.get_feature_names_out())
processed['stroke'] = y.to_numpy()
print('Processed shape:', processed.shape)
processed.head()

## Simpan deliverable
`dataset_clean.csv` berisi data yang sudah diaudit dan diimputasi. `dataset_preprocessed.csv` berisi fitur hasil encoding + scaling yang siap dipakai untuk pemodelan, dengan `stroke` sebagai target.

In [ ]:
df.to_csv('dataset_clean.csv', index=False)
processed.to_csv('dataset_preprocessed.csv', index=False)
print('Saved: dataset_clean.csv, dataset_preprocessed.csv')

## Ringkasan keputusan
Data CSV dimuat ke Pandas lalu diaudit untuk duplikasi, tipe, dan missing value. Hanya `bmi` yang memiliki missing value, sehingga median dipilih sebagai imputasi yang robust; kategori `Unknown` pada `smoking_status` dipertahankan sebagai informasi tersendiri. Fitur kontinu distandardisasi, fitur biner dipertahankan sebagai 0/1, dan fitur nominal di-One-Hot Encode menggunakan `ColumnTransformer` agar preprocessing konsisten dan dapat digunakan kembali pada data baru.